# CRATED BY ER.GAURAV SAPKAR 
# SQL Practical Record (Corrected) — Runnable in Jupyter
### EMP / DEPT / PROJECT Tables — Queries, DML, Transactions & Privileges
> All typos and invalid identifiers have been fixed. Every query below actually runs — executed here using Python's built-in `sqlite3` engine (since Jupyter has no native Oracle server).
>
> The **original Oracle SQL** is always shown first. Where Oracle syntax needed a small tweak to run on SQLite (e.g. `DATE '2026-03-01'` -> `'2026-03-01'`, or `GRANT`/`REVOKE` which SQLite doesn't support at all), that is noted before the runnable version.

## Setup — create EMP and DEPT tables (the classic SCOTT schema) with sample data

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

cur.executescript("""
CREATE TABLE DEPT (
  DEPTNO INTEGER PRIMARY KEY,
  DNAME TEXT,
  LOC TEXT
);
INSERT INTO DEPT VALUES (10,'ACCOUNTING','NEW YORK');
INSERT INTO DEPT VALUES (20,'RESEARCH','DALLAS');
INSERT INTO DEPT VALUES (30,'SALES','CHICAGO');
INSERT INTO DEPT VALUES (40,'OPERATIONS','BOSTON');

CREATE TABLE EMP (
  EMPNO INTEGER PRIMARY KEY,
  ENAME TEXT,
  JOB TEXT,
  MGR INTEGER,
  HIREDATE TEXT,
  SAL REAL,
  COMM REAL,
  DEPTNO INTEGER
);
INSERT INTO EMP VALUES (7369,'SMITH','CLERK',7902,'17-DEC-80',800,NULL,20);
INSERT INTO EMP VALUES (7499,'ALLEN','SALESMAN',7698,'20-FEB-81',1600,300,30);
INSERT INTO EMP VALUES (7521,'WARD','SALESMAN',7698,'22-FEB-81',1250,500,30);
INSERT INTO EMP VALUES (7566,'JONES','MANAGER',7839,'02-APR-81',2975,NULL,20);
INSERT INTO EMP VALUES (7654,'MARTIN','SALESMAN',7698,'28-SEP-81',1250,1400,30);
INSERT INTO EMP VALUES (7698,'BLAKE','MANAGER',7839,'01-MAY-81',2850,NULL,30);
INSERT INTO EMP VALUES (7782,'CLARK','MANAGER',7839,'09-JUN-81',2450,NULL,10);
INSERT INTO EMP VALUES (7788,'SCOTT','ANALYST',7566,'19-APR-87',3000,NULL,20);
INSERT INTO EMP VALUES (7839,'KING','PRESIDENT',NULL,'17-NOV-81',5000,NULL,10);
INSERT INTO EMP VALUES (7844,'TURNER','SALESMAN',7698,'08-SEP-81',1500,0,30);
INSERT INTO EMP VALUES (7876,'ADAMS','CLERK',7788,'23-MAY-87',1100,NULL,20);
INSERT INTO EMP VALUES (7900,'JAMES','CLERK',7698,'03-DEC-81',950,NULL,30);
INSERT INTO EMP VALUES (7902,'FORD','ANALYST',7566,'03-DEC-81',3000,NULL,20);
INSERT INTO EMP VALUES (7934,'MILLER','CLERK',7782,'23-JAN-82',1300,NULL,10);
""")
conn.commit()
print("EMP and DEPT tables ready.")

def run_select(query):
    """Run a SELECT and display it as a dataframe."""
    df = pd.read_sql_query(query, conn)
    print(f"{len(df)} row(s) selected.")
    return df

def run_dml(query, success_msg):
    """Run an INSERT/UPDATE/DELETE/DDL statement (no auto-commit, so SAVEPOINT/ROLLBACK work as expected)."""
    cur.execute(query)
    print(success_msg, f"(rowcount={cur.rowcount})")


EMP and DEPT tables ready.


## Question 1
**Q:** Display the employee name and employee number of employees whose name starts with 'S' and has 'I' as the third letter from the end.

> **Fixed:** Corrected column name EMPID -> EMPNO (EMP table has no EMPID column).

In [2]:
# --- Original Oracle SQL ---
# SELECT ENAME, EMPNO
# FROM EMP
# WHERE ENAME LIKE 'S%'
#   AND SUBSTR(ENAME, LENGTH(ENAME)-2, 1) = 'I';

query = """
SELECT ENAME, EMPNO
FROM EMP
WHERE ENAME LIKE 'S%'
  AND SUBSTR(ENAME, LENGTH(ENAME)-2, 1) = 'I'
"""
run_select(query)

1 row(s) selected.


,ENAME,EMPNO
0,SMITH,7369


## Question 2
**Q:** Display the employee name, salary, and annual salary (salary * 12) for employees whose annual salary is greater than 36000.

In [3]:
# --- Original Oracle SQL ---
# SELECT ENAME, SAL, (SAL * 12) AS ANNUAL_SALARY
# FROM EMP
# WHERE (SAL * 12) > 36000;

query = """
SELECT ENAME, SAL, (SAL * 12) AS ANNUAL_SALARY
FROM EMP
WHERE (SAL * 12) > 36000
"""
run_select(query)

1 row(s) selected.


,ENAME,SAL,ANNUAL_SALARY
0,KING,5000.0,60000.0


## Question 3
**Q:** Display all details of employees who belong to department 10 or 20 and are not managers.

In [4]:
# --- Original Oracle SQL ---
# SELECT *
# FROM EMP
# WHERE DEPTNO IN (10, 20)
#   AND JOB <> 'MANAGER';

query = """
SELECT *
FROM EMP
WHERE DEPTNO IN (10, 20)
  AND JOB <> 'MANAGER'
"""
run_select(query)

6 row(s) selected.


,EMPNO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,7902.0,17-DEC-80,800.0,None,20
1,7788,SCOTT,ANALYST,7566.0,19-APR-87,3000.0,None,20
2,7839,KING,PRESIDENT,NaN,17-NOV-81,5000.0,None,10
3,7876,ADAMS,CLERK,7788.0,23-MAY-87,1100.0,None,20
4,7902,FORD,ANALYST,7566.0,03-DEC-81,3000.0,None,20
5,7934,MILLER,CLERK,7782.0,23-JAN-82,1300.0,None,10


## Question 4
**Q:** After increasing the line/page size (SET LINES 5000 PAGES 5000), display all employee records.

In [5]:
# --- Original Oracle SQL ---
# SET LINES 5000 PAGES 5000
# SELECT * FROM EMP;

# NOTE: "SET LINES/PAGES" is an SQL*Plus display setting, not a SQL statement -
# it has no SQLite equivalent and is skipped here; only the SELECT actually runs.
query = """
SELECT * FROM EMP
"""
run_select(query)

14 row(s) selected.


,EMPNO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7369,SMITH,CLERK,7902.0,17-DEC-80,800.0,NaN,20
1,7499,ALLEN,SALESMAN,7698.0,20-FEB-81,1600.0,300.0,30
2,7521,WARD,SALESMAN,7698.0,22-FEB-81,1250.0,500.0,30
3,7566,JONES,MANAGER,7839.0,02-APR-81,2975.0,NaN,20
4,7654,MARTIN,SALESMAN,7698.0,28-SEP-81,1250.0,1400.0,30
5,7698,BLAKE,MANAGER,7839.0,01-MAY-81,2850.0,NaN,30
6,7782,CLARK,MANAGER,7839.0,09-JUN-81,2450.0,NaN,10
7,7788,SCOTT,ANALYST,7566.0,19-APR-87,3000.0,NaN,20
8,7839,KING,PRESIDENT,NaN,17-NOV-81,5000.0,NaN,10
9,7844,TURNER,SALESMAN,7698.0,08-SEP-81,1500.0,0.0,30


## Question 5
**Q:** Display all details of employees whose job is CLERK or ANALYST and whose salary is between 1000 and 3000.

In [6]:
# --- Original Oracle SQL ---
# SELECT *
# FROM EMP
# WHERE JOB IN ('CLERK', 'ANALYST')
#   AND SAL BETWEEN 1000 AND 3000;

query = """
SELECT *
FROM EMP
WHERE JOB IN ('CLERK', 'ANALYST')
  AND SAL BETWEEN 1000 AND 3000
"""
run_select(query)

4 row(s) selected.


,EMPNO,ENAME,JOB,MGR,HIREDATE,SAL,COMM,DEPTNO
0,7788,SCOTT,ANALYST,7566,19-APR-87,3000.0,None,20
1,7876,ADAMS,CLERK,7788,23-MAY-87,1100.0,None,20
2,7902,FORD,ANALYST,7566,03-DEC-81,3000.0,None,20
3,7934,MILLER,CLERK,7782,23-JAN-82,1300.0,None,10


## Question 6
**Q:** Display the employee name, job, and commission for employees who have no commission (NULL or 0).

In [7]:
# --- Original Oracle SQL ---
# SELECT ENAME, JOB, COMM
# FROM EMP
# WHERE COMM IS NULL OR COMM = 0;

query = """
SELECT ENAME, JOB, COMM
FROM EMP
WHERE COMM IS NULL OR COMM = 0
"""
run_select(query)

11 row(s) selected.


,ENAME,JOB,COMM
0,SMITH,CLERK,NaN
1,JONES,MANAGER,NaN
2,BLAKE,MANAGER,NaN
3,CLARK,MANAGER,NaN
4,SCOTT,ANALYST,NaN
5,KING,PRESIDENT,NaN
6,TURNER,SALESMAN,0.0
7,ADAMS,CLERK,NaN
8,JAMES,CLERK,NaN
9,FORD,ANALYST,NaN


## Question 7
**Q:** Display the department name and location for departments located in New York.

In [8]:
# --- Original Oracle SQL ---
# SELECT DNAME, LOC
# FROM DEPT
# WHERE LOC = 'NEW YORK';

query = """
SELECT DNAME, LOC
FROM DEPT
WHERE LOC = 'NEW YORK'
"""
run_select(query)

1 row(s) selected.


,DNAME,LOC
0,ACCOUNTING,NEW YORK


## Question 8
**Q:** Display the employee name and salary for employees whose name starts with 'S' or ends with 'N', and whose salary is greater than 2000.

In [9]:
# --- Original Oracle SQL ---
# SELECT ENAME, SAL
# FROM EMP
# WHERE (ENAME LIKE 'S%' OR ENAME LIKE '%N')
#   AND SAL > 2000;

query = """
SELECT ENAME, SAL
FROM EMP
WHERE (ENAME LIKE 'S%' OR ENAME LIKE '%N')
  AND SAL > 2000
"""
run_select(query)

1 row(s) selected.


,ENAME,SAL
0,SCOTT,3000.0


## Question 9
**Q:** Create a PROJECT table with columns PROJ_ID (primary key), PROJ_NAME, START_DATE, and END_DATE.

In [10]:
# --- Original Oracle SQL ---
# CREATE TABLE PROJECT (
#     PROJ_ID    NUMBER PRIMARY KEY,
#     PROJ_NAME  VARCHAR2(20),
#     START_DATE DATE,
#     END_DATE   DATE
# );

cur.execute("""
CREATE TABLE PROJECT (
    PROJ_ID    INTEGER PRIMARY KEY,
    PROJ_NAME  TEXT,
    START_DATE TEXT,
    END_DATE   TEXT
);
""")
conn.commit()
print("Table created.")

Table created.


## Question 10
**Q:** Display the employee name and annual salary (salary * 12) for employees whose job is MANAGER or CLERK.

> **Fixed:** Removed the stray, mistyped "ED" line that had left the original statement incomplete.

In [11]:
# --- Original Oracle SQL ---
# SELECT ENAME, SAL * 12 AS ANNUAL_SALARY
# FROM EMP
# WHERE JOB IN ('MANAGER', 'CLERK');

query = """
SELECT ENAME, SAL * 12 AS ANNUAL_SALARY
FROM EMP
WHERE JOB IN ('MANAGER', 'CLERK')
"""
run_select(query)

7 row(s) selected.


,ENAME,ANNUAL_SALARY
0,SMITH,9600.0
1,JONES,35700.0
2,BLAKE,34200.0
3,CLARK,29400.0
4,ADAMS,13200.0
5,JAMES,11400.0
6,MILLER,15600.0


## Question 11
**Q:** Display the name, job, and salary of all employees.

In [12]:
# --- Original Oracle SQL ---
# SELECT ENAME, JOB, SAL
# FROM EMP;

query = """
SELECT ENAME, JOB, SAL
FROM EMP
"""
run_select(query)

14 row(s) selected.


,ENAME,JOB,SAL
0,SMITH,CLERK,800.0
1,ALLEN,SALESMAN,1600.0
2,WARD,SALESMAN,1250.0
3,JONES,MANAGER,2975.0
4,MARTIN,SALESMAN,1250.0
5,BLAKE,MANAGER,2850.0
6,CLARK,MANAGER,2450.0
7,SCOTT,ANALYST,3000.0
8,KING,PRESIDENT,5000.0
9,TURNER,SALESMAN,1500.0


## Question 12
**Q:** Insert a new project record with ID 101 ('AI System').

In [13]:
# --- Original Oracle SQL ---
# INSERT INTO PROJECT VALUES (101, 'AI System', DATE '2026-03-01', DATE '2026-06-01');

# --- Adapted for SQLite (DATE literal -> plain string) ---
run_dml("""INSERT INTO PROJECT VALUES (101, 'AI System', '2026-03-01', '2026-06-01')""", "1 row created.")

1 row created. (rowcount=1)


## Question 13
**Q:** Insert a new project record with ID 102 ('Banking App').

In [14]:
# --- Original Oracle SQL ---
# INSERT INTO PROJECT VALUES (102, 'Banking App', DATE '2026-04-01', DATE '2026-07-01');

# --- Adapted for SQLite (DATE literal -> plain string) ---
run_dml("""INSERT INTO PROJECT VALUES (102, 'Banking App', '2026-04-01', '2026-07-01')""", "1 row created.")

1 row created. (rowcount=1)


## Question 14
**Q:** Update the end date of the project with ID 102.

In [15]:
# --- Original Oracle SQL ---
# UPDATE PROJECT
# SET END_DATE = DATE '2026-08-01'
# WHERE PROJ_ID = 102;

# --- Adapted for SQLite (DATE literal -> plain string) ---
run_dml("""UPDATE PROJECT
SET END_DATE = '2026-08-01'
WHERE PROJ_ID = 102""", "1 row updated.")

1 row updated. (rowcount=1)


## Question 15
**Q:** Delete the project record with ID 101.

In [16]:
# --- Original Oracle SQL ---
# DELETE FROM PROJECT
# WHERE PROJ_ID = 101;

run_dml("""DELETE FROM PROJECT
WHERE PROJ_ID = 101""", "1 row deleted.")

1 row deleted. (rowcount=1)


## Question 16
**Q:** Display all records from the PROJECT table.

> **Fixed:** Corrected the typo "FORM" -> "FROM".

In [17]:
# --- Original Oracle SQL ---
# SELECT * FROM PROJECT;

query = """
SELECT * FROM PROJECT
"""
run_select(query)

1 row(s) selected.


,PROJ_ID,PROJ_NAME,START_DATE,END_DATE
0,102,Banking App,2026-04-01,2026-08-01


## Question 17
**Q:** Insert a project record (ID 101, 'AI Research') using explicit column names.

> **Fixed:** Corrected column names Project_ID/Project_Name -> PROJ_ID/PROJ_NAME (matching the table definition) and used proper DATE literals.

In [18]:
# --- Original Oracle SQL ---
# INSERT INTO PROJECT (PROJ_ID, PROJ_NAME, START_DATE, END_DATE)
# VALUES (101, 'AI Research', DATE '2026-03-01', DATE '2026-12-31');

# --- Adapted for SQLite (DATE literal -> plain string) ---
run_dml("""INSERT INTO PROJECT (PROJ_ID, PROJ_NAME, START_DATE, END_DATE)
VALUES (101, 'AI Research', '2026-03-01', '2026-12-31')""", "1 row created.")

1 row created. (rowcount=1)


## Question 18
**Q:** Create a savepoint named sp1.

In [19]:
# --- Original Oracle SQL ---
# SAVEPOINT sp1;

cur.execute("SAVEPOINT sp1;")
print("Savepoint created.")

Savepoint created.


## Question 19
**Q:** Update the project name for the record with ID 101.

> **Fixed:** Corrected column names Project_Name/Project_ID -> PROJ_NAME/PROJ_ID.

In [20]:
# --- Original Oracle SQL ---
# UPDATE PROJECT
# SET PROJ_NAME = 'AI Research Updated'
# WHERE PROJ_ID = 101;

run_dml("""UPDATE PROJECT
SET PROJ_NAME = 'AI Research Updated'
WHERE PROJ_ID = 101""", "1 row updated.")

1 row updated. (rowcount=1)


## Question 20
**Q:** Roll back the transaction to savepoint sp1.

In [21]:
# --- Original Oracle SQL ---
# ROLLBACK TO sp1;

cur.execute("ROLLBACK TO sp1;")
print("Rollback complete.")

Rollback complete.


## Question 21
**Q:** Commit the current transaction.

In [22]:
# --- Original Oracle SQL ---
# COMMIT;

conn.commit()
print("Commit complete.")

Commit complete.


## Question 22
**Q:** Grant SELECT and INSERT privileges on the PROJECT table to user HR.

In [23]:
# --- Original Oracle SQL ---
# GRANT SELECT, INSERT
# ON PROJECT
# TO HR;

# NOTE: SQLite has no concept of database users/privileges, so GRANT/REVOKE
# cannot actually be executed here. This is genuine Oracle-only syntax - shown
# for completeness, simulating what Oracle would report.
print("Grant succeeded.   -- (simulated; SQLite has no user/privilege model)")

Grant succeeded.   -- (simulated; SQLite has no user/privilege model)


## Question 23
**Q:** Revoke the INSERT privilege on the PROJECT table from user HR.

In [24]:
# --- Original Oracle SQL ---
# REVOKE INSERT
# ON PROJECT
# FROM HR;

# See note on Q22 - GRANT/REVOKE are Oracle privilege statements, not
# something SQLite can execute.
print("Revoke succeeded.  -- (simulated; SQLite has no user/privilege model)")

Revoke succeeded.  -- (simulated; SQLite has no user/privilege model)


## Final check — state of the PROJECT table after all operations

In [25]:
run_select("SELECT * FROM PROJECT")

2 row(s) selected.


,PROJ_ID,PROJ_NAME,START_DATE,END_DATE
0,101,AI Research,2026-03-01,2026-12-31
1,102,Banking App,2026-04-01,2026-08-01


------------------------------------------------------END-------------------------------------------------------------------

|